# Transformer Architecture: Efficient Attention, Mixture of Experts, and Adaptation

This notebook covers advanced transformer techniques:

1. **Group Query Attention (GQA)** — reducing KV cache memory by sharing keys/values across query heads
2. **Sliding Window Attention** — reducing quadratic compute cost to linear via local windows
3. **Mixture of Experts (MoE)** — decoupling parameter count from active compute
4. **In-Context Learning (ICL)** — few-shot and zero-shot prompting without parameter updates
5. **Supervised Fine-Tuning (SFT) / Instruction Tuning** — adapting a pretrained model to instruction-following


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## 1. Group Query Attention (GQA)

### 1.1 Motivation: KV Cache Bottleneck

During autoregressive generation at time step $t$, the transformer must retain keys and values from all previous positions for every attention head. Standard **Multi-Head Attention (MHA)** with $N_H$ heads stores:

$$\text{KV cache size} \propto N_H \times t \times 2 \times d_H$$

This grows linearly in sequence length $t$ and head count $N_H$, limiting the batch size that fits in GPU memory — which in turn limits inference throughput.

### 1.2 The GQA Idea

Keep the full $N_H$ query heads but share a much smaller set of $N_G$ key/value groups where $N_G \ll N_H$. Define the integer ratio $\tau = N_H / N_G$ (assumed to divide evenly). Head $j$ (1-indexed) is mapped to group:

$$g(j) = \left\lfloor \frac{j-1}{\tau} \right\rfloor + 1$$

The attention for head $j$ becomes:

$$\text{Attn}_j = \text{softmax}\!\left(\frac{Q_j \, K_{g(j)}^\top + M}{\sqrt{d_H}}\right) V_{g(j)}$$

where $M$ is the causal mask. With $N_G = 1$ this is **Multi-Query Attention (MQA)**; with $N_G = N_H$ it recovers standard MHA.

| Variant | $N_G$ | KV cache scaling |
|---------|-------|------------------|
| MHA | $N_H$ | $N_H \times t$ |
| GQA | $1 < N_G < N_H$ | $N_G \times t$ |
| MQA | $1$ | $1 \times t$ |


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q: (T, d_H), K: (T, d_H), V: (T, d_H)"""
    d_H = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_H)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    probs = np.exp(scores - scores.max(axis=-1, keepdims=True))
    probs /= probs.sum(axis=-1, keepdims=True)
    return probs @ V, probs


def group_query_attention(X, W_Q, W_K, W_V, N_H, N_G):
    """
    X   : (T, d_model)
    W_Q : (N_H, d_model, d_H)  — one projection per query head
    W_K : (N_G, d_model, d_H)  — one projection per key group
    W_V : (N_G, d_model, d_H)  — one projection per value group
    Returns list of (output, attn_weights) per query head.
    """
    T = X.shape[0]
    tau = N_H // N_G
    causal_mask = np.tril(np.ones((T, T), dtype=bool))

    K_groups = [X @ W_K[g] for g in range(N_G)]
    V_groups = [X @ W_V[g] for g in range(N_G)]

    results = []
    for j in range(N_H):
        g = j // tau
        Q_j = X @ W_Q[j]
        out, attn = scaled_dot_product_attention(Q_j, K_groups[g], V_groups[g], causal_mask)
        results.append((out, attn))
    return results


T, d_model, d_H = 8, 16, 4
N_H_demo, N_G_demo = 4, 2
tau_demo = N_H_demo // N_G_demo

X_demo = np.random.randn(T, d_model)
W_Q_demo = np.random.randn(N_H_demo, d_model, d_H) * 0.1
W_K_demo = np.random.randn(N_G_demo, d_model, d_H) * 0.1
W_V_demo = np.random.randn(N_G_demo, d_model, d_H) * 0.1

head_results = group_query_attention(X_demo, W_Q_demo, W_K_demo, W_V_demo, N_H_demo, N_G_demo)

fig, axes = plt.subplots(1, N_H_demo, figsize=(12, 3))
colors = ['Blues', 'Greens', 'Oranges', 'Purples']
for j, (_, attn) in enumerate(head_results):
    g = j // tau_demo
    axes[j].imshow(attn, cmap=colors[j], vmin=0, vmax=1)
    axes[j].set_title(f'Head {j+1}\n(KV group {g+1})', fontsize=10)
    axes[j].set_xlabel('Key position')
    axes[j].set_ylabel('Query position')

plt.suptitle(f'GQA causal attention maps — {N_H_demo} query heads, {N_G_demo} KV groups', fontsize=12)
plt.tight_layout()
plt.show()

print(f"N_H={N_H_demo}, N_G={N_G_demo}, tau={tau_demo}")
print(f"KV cache reduction factor vs MHA: {N_H_demo / N_G_demo:.1f}x")

### 1.3 KV Cache Size vs. Number of Groups

The KV cache grows linearly in $N_G$. The plot below shows how GQA interpolates between full MHA memory cost and the minimal MQA cost.

In [ ]:
N_H_vals = [8, 16, 32, 64]
T_seq = 4096
d_H_size = 128

fig, axes = plt.subplots(1, len(N_H_vals), figsize=(14, 4))
for ax, NH in zip(axes, N_H_vals):
    divisors = [ng for ng in range(1, NH + 1) if NH % ng == 0]
    kv_sizes = [ng * T_seq * d_H_size * 2 / 1e6 for ng in divisors]
    ax.plot(divisors, kv_sizes, 'o-', color='steelblue')
    ax.axhline(kv_sizes[-1], color='red', linestyle='--', alpha=0.6, label='MHA')
    ax.axhline(kv_sizes[0], color='green', linestyle='--', alpha=0.6, label='MQA')
    ax.set_xlabel('N_G (KV groups)')
    ax.set_ylabel('KV cache (M floats)')
    ax.set_title(f'N_H = {NH}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'KV cache size vs. number of KV groups (T={T_seq}, d_H={d_H_size})', fontsize=12)
plt.tight_layout()
plt.show()

### 1.4 Query-Key Normalization

The softmax input is the inner product $Q_j K_{g(j)}^\top / \sqrt{d_H}$. If the entries of $Q$ and $K$ are $O(1)$, each inner product has standard deviation $\approx \sqrt{d_H}$, so dividing by $\sqrt{d_H}$ keeps the pre-softmax scale $O(1)$ regardless of $d_H$.

An alternative used in some models is to apply RMS-norm to $Q$ and $K$ before the inner product:

$$\text{Attn}_j = \text{softmax}\!\left(\frac{\text{RMSNorm}(Q_j)\,\text{RMSNorm}(K_{g(j)})^\top + M}{\sqrt{d_H}}\right) V_{g(j)}$$

This makes the scale robust to changes in weight magnitude during training.

In [ ]:
def rms_norm(x, eps=1e-6):
    return x / (np.sqrt(np.mean(x**2, axis=-1, keepdims=True)) + eps)


d_H_vals = [4, 16, 64, 256]
n_trials = 5000

std_raw, std_normed = [], []
for dh in d_H_vals:
    a = np.random.randn(n_trials, dh)
    b = np.random.randn(n_trials, dh)
    raw = (a * b).sum(axis=1) / np.sqrt(dh)
    a_n, b_n = rms_norm(a), rms_norm(b)
    normed = (a_n * b_n).sum(axis=1) / np.sqrt(dh)
    std_raw.append(raw.std())
    std_normed.append(normed.std())

x_pos = np.arange(len(d_H_vals))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_pos - 0.2, std_raw, 0.4, label='Without RMSNorm', color='salmon')
ax.bar(x_pos + 0.2, std_normed, 0.4, label='With RMSNorm', color='steelblue')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(d) for d in d_H_vals])
ax.set_xlabel('Head dimension d_H')
ax.set_ylabel('Std of scaled inner product')
ax.set_title('Effect of pre-normalization on attention logit scale')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("d_H  | std (raw) | std (RMSNorm)")
for dh, sr, sn in zip(d_H_vals, std_raw, std_normed):
    print(f"{dh:4d} | {sr:.3f}     | {sn:.3f}")

---
## 2. Sliding Window Attention

### 2.1 Quadratic Cost in Standard Attention

In standard causal attention over a sequence of length $T$, each query attends to all past keys, giving $O(T^2)$ compute and $O(T)$ KV cache memory. For $T = 10^6$, the $T^2$ term becomes intractable.

### 2.2 Local Window Attention

**Sliding window attention** restricts each position $t$ to attend only to the $W$ most recent positions:

$$\text{Attend}(Q_t, \{K_s\}_{s=\max(1,t-W)}^t, \{V_s\}_{s=\max(1,t-W)}^t)$$

Compute drops to $O(T \cdot W)$, which is linear in $T$ when $W$ is fixed.

### 2.3 Receptive Field Through Depth

With $L$ layers each with window $W$, position $t$ in layer $L$ has an effective receptive field of $L \cdot W$ tokens — the hidden representation at any position carries information about up to $W$ earlier positions from the layer below, and so on recursively. This allows long-range information propagation without paying quadratic cost, at the expense of losing information beyond $L \cdot W$ tokens.

In [ ]:
def sliding_window_mask(T, W):
    mask = np.zeros((T, T), dtype=bool)
    for t in range(T):
        lo = max(0, t - W + 1)
        mask[t, lo:t + 1] = True
    return mask


def full_causal_mask(T):
    return np.tril(np.ones((T, T), dtype=bool))


T_demo = 16
windows = [2, 4, 8, T_demo]

fig, axes = plt.subplots(1, len(windows), figsize=(13, 3.5))
for ax, W in zip(axes, windows):
    mask = sliding_window_mask(T_demo, W)
    ax.imshow(mask.astype(float), cmap='Blues', vmin=0, vmax=1)
    active = mask.sum()
    total = T_demo * (T_demo + 1) // 2
    ax.set_title(f'W={W}\n{active}/{total} pairs attended', fontsize=9)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')

plt.suptitle(f'Sliding window attention masks (T={T_demo})', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
T_range = np.arange(100, 20001, 500)
W_fixed = 512

compute_full = T_range**2
compute_sliding = T_range * W_fixed

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(T_range, compute_full / 1e6, label='Full causal ($T^2$)', color='red')
axes[0].plot(T_range, compute_sliding / 1e6, label=f'Sliding window ($T \\cdot W$, W={W_fixed})', color='steelblue')
axes[0].set_xlabel('Sequence length T')
axes[0].set_ylabel('Attention pairs (millions)')
axes[0].set_title('Compute scaling')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

L_vals = [2, 4, 8, 16, 32]
for W in [128, 256, 512, 1024]:
    receptive = [L * W for L in L_vals]
    axes[1].plot(L_vals, [r / 1000 for r in receptive], marker='o', label=f'W={W}')

axes[1].set_xlabel('Number of layers L')
axes[1].set_ylabel('Effective receptive field (k tokens)')
axes[1].set_title('Receptive field = L × W')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Mixture of Experts (MoE)

### 3.1 Decoupling Parameters from Active Compute

In a standard transformer the MLP block at each position uses all parameters. In a **Mixture of Experts** layer:

- There are $M$ expert MLPs $\{E_1, \ldots, E_M\}$
- A router selects the $K$ highest-scoring experts per token
- Only those $K$ experts execute; the rest do not contribute to compute

This means total parameters $\propto M$ but active compute $\propto K$ per token. A "30B A3B" model has 30B total parameters and 3B active per token.

### 3.2 Routing

Given hidden vector $h \in \mathbb{R}^d$, a router computes a relevance score for each expert:

$$r = \sigma(W_r h) \in \mathbb{R}^M$$

where $\sigma$ can be softmax or sigmoid. The selected expert set is the $K$ indices with highest $r$:

$$S = \text{TopK}(r, K)$$

The output is a weighted sum of the selected experts, with weights renormalized over $S$:

$$\text{out} = \sum_{s \in S} \alpha_s \, E_s(h), \quad \alpha_s = \frac{r_s}{\sum_{s' \in S} r_{s'}}$$

### 3.3 Load Balancing

If routing collapses onto a few experts, most expert capacity is wasted and those experts become hotspots on specific GPUs. A **load-balancing auxiliary loss** encourages uniform expert utilization:

$$\mathcal{L}_{\text{aux}} = M \sum_{i=1}^{M} f_i \cdot p_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ and $p_i$ is the mean router probability for expert $i$.

In [ ]:
class MoELayer:
    def __init__(self, d_model, d_expert, M, K):
        self.M = M
        self.K = K
        self.W_router = np.random.randn(d_model, M) * 0.1
        self.experts = [
            {'W1': np.random.randn(d_model, d_expert) * 0.1,
             'W2': np.random.randn(d_expert, d_model) * 0.1}
            for _ in range(M)
        ]

    def _expert_forward(self, e_idx, h):
        W1, W2 = self.experts[e_idx]['W1'], self.experts[e_idx]['W2']
        return np.maximum(0, h @ W1) @ W2

    def forward(self, H):
        """H: (T, d_model) — process a batch of tokens"""
        T = H.shape[0]
        logits = H @ self.W_router
        r = np.exp(logits - logits.max(axis=-1, keepdims=True))
        r /= r.sum(axis=-1, keepdims=True)

        top_k_idx = np.argsort(r, axis=-1)[:, -self.K:]

        out = np.zeros_like(H)
        expert_counts = np.zeros(self.M)
        for t in range(T):
            selected = top_k_idx[t]
            weights = r[t, selected]
            weights = weights / weights.sum()
            for s, w in zip(selected, weights):
                out[t] += w * self._expert_forward(s, H[t])
                expert_counts[s] += 1
        return out, r, expert_counts


d_model, d_expert, M_experts, K_active = 32, 64, 8, 2
T_tokens = 64

moe = MoELayer(d_model, d_expert, M_experts, K_active)
H_in = np.random.randn(T_tokens, d_model)
H_out, router_probs, expert_counts = moe.forward(H_in)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(router_probs.T, aspect='auto', cmap='viridis')
axes[0].set_xlabel('Token')
axes[0].set_ylabel('Expert')
axes[0].set_title('Router probabilities (T x M)')
plt.colorbar(axes[0].images[0], ax=axes[0])

axes[1].bar(range(M_experts), expert_counts, color='steelblue')
axes[1].axhline(T_tokens * K_active / M_experts, color='red', linestyle='--', label='Uniform load')
axes[1].set_xlabel('Expert index')
axes[1].set_ylabel('Tokens routed')
axes[1].set_title(f'Expert utilization (K={K_active}, M={M_experts})')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].hist(expert_counts, bins=range(0, int(expert_counts.max()) + 2), color='salmon', edgecolor='black')
axes[2].set_xlabel('Tokens routed to expert')
axes[2].set_ylabel('Number of experts')
axes[2].set_title('Load distribution')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Expert utilization: {expert_counts}")
print(f"Expected uniform load: {T_tokens * K_active / M_experts:.1f} tokens/expert")
print(f"Load imbalance (std/mean): {expert_counts.std() / expert_counts.mean():.3f}")

### 3.4 Load Balancing Auxiliary Loss

The auxiliary loss penalizes imbalance by correlating the fraction of tokens routed to each expert $f_i$ with the mean router probability $p_i$ for that expert. With perfect balance $f_i = p_i = 1/M$ and the loss equals 1.

In [ ]:
def aux_load_balance_loss(router_probs, top_k_idx, M):
    T = router_probs.shape[0]
    expert_mask = np.zeros((T, M))
    for t in range(T):
        expert_mask[t, top_k_idx[t]] = 1.0
    f = expert_mask.mean(axis=0)
    p = router_probs.mean(axis=0)
    return M * np.dot(f, p), f, p


T_tok = 128
M_e = 8
K_a = 2
n_seeds = 8

losses, imbalances = [], []
fig, axes = plt.subplots(2, n_seeds // 2, figsize=(14, 5), sharex=False)
axes = axes.flatten()

for seed_i in range(n_seeds):
    np.random.seed(seed_i * 7)
    W_r = np.random.randn(d_model, M_e) * (0.05 + seed_i * 0.15)
    H = np.random.randn(T_tok, d_model)
    logits = H @ W_r
    probs = np.exp(logits - logits.max(axis=-1, keepdims=True))
    probs /= probs.sum(axis=-1, keepdims=True)
    top_k = np.argsort(probs, axis=-1)[:, -K_a:]
    loss, f, p = aux_load_balance_loss(probs, top_k, M_e)
    counts = np.zeros(M_e)
    for t in range(T_tok):
        counts[top_k[t]] += 1
    losses.append(loss)
    imbalances.append(counts.std() / counts.mean())
    axes[seed_i].bar(range(M_e), counts / T_tok, color='steelblue', alpha=0.8)
    axes[seed_i].axhline(K_a / M_e, color='red', linestyle='--', linewidth=1)
    axes[seed_i].set_title(f'Loss={loss:.3f}', fontsize=9)
    axes[seed_i].set_ylim(0, 1)
    axes[seed_i].tick_params(labelsize=7)

plt.suptitle('Load distributions for different router weight scales\n(red line = uniform)', fontsize=11)
plt.tight_layout()
plt.show()

print("Loss values:", [f"{l:.3f}" for l in losses])
print("Imbalance (std/mean):", [f"{v:.3f}" for v in imbalances])

---
## 4. In-Context Learning (ICL)

### 4.1 How It Works

A pretrained language model $p_\theta$ can perform a new task without any parameter updates. The key idea: **the task specification is encoded entirely in the input context**.

Given examples $\{(x_1, y_1), \ldots, (x_n, y_n)\}$ and test input $x_{n+1}$, the model processes:

```
[Q: x_1  A: y_1]  [Q: x_2  A: y_2]  ...  [Q: x_n  A: y_n]  [Q: x_test  A: ???]
```

and generates $y_{n+1}$ by sampling from $p_\theta(y \mid \text{context})$. Model parameters $\theta$ are **never updated**.

### 4.2 Few-Shot vs. Zero-Shot

| Mode | Examples provided | Parameter update |
|------|-------------------|------------------|
| Zero-shot | 0 (task description only) | None |
| Few-shot ($k$-shot) | $k$ labeled examples | None |
| Fine-tuning | Full dataset | Yes |

Zero-shot requires the model to generalize from a natural language task description alone. Few-shot provides $k$ demonstrations so the model can infer the format, mapping, and output style.

In [ ]:
def simulate_icl_accuracy(n_shots_range, task_difficulty=1.0, base_accuracy=0.55, ceiling=0.92, seed=0):
    """
    Toy model of how few-shot accuracy improves with number of examples.
    Uses a saturating learning curve: acc = ceiling - (ceiling - base) * exp(-k * n_shots / difficulty)
    """
    np.random.seed(seed)
    k_rate = 0.4
    smooth = ceiling - (ceiling - base_accuracy) * np.exp(-k_rate * np.array(n_shots_range) / task_difficulty)
    noise = np.random.randn(len(n_shots_range)) * 0.015
    return np.clip(smooth + noise, 0, 1)


n_shots = [0, 1, 2, 4, 8, 16, 32]
tasks = [
    ('Simple classification', 1.0, 0.65, 0.94),
    ('Domain-specific labeling', 2.5, 0.52, 0.88),
    ('Symbolic arithmetic', 4.0, 0.45, 0.82),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, difficulty, base, ceiling in tasks:
    acc = simulate_icl_accuracy(n_shots, difficulty, base, ceiling)
    axes[0].plot(n_shots, acc * 100, marker='o', label=name)

axes[0].set_xlabel('Number of in-context examples (shots)')
axes[0].set_ylabel('Task accuracy (%)')
axes[0].set_title('Few-shot accuracy vs. task difficulty')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(n_shots)

categories = ['Zero-shot\n(description only)', '1-shot', '4-shot', '16-shot', 'Fine-tuned']
easy_acc = [65, 75, 85, 90, 93]
hard_acc = [45, 55, 65, 73, 91]

x_pos = np.arange(len(categories))
width = 0.35
axes[1].bar(x_pos - width/2, easy_acc, width, label='Simple task', color='steelblue', alpha=0.85)
axes[1].bar(x_pos + width/2, hard_acc, width, label='Difficult task', color='salmon', alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(categories, fontsize=9)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('ICL vs. fine-tuning (schematic comparison)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 4.3 What ICL Can and Cannot Do

ICL adapts **style and output format** from examples without updating parameters. It is effective for:
- Learning label names and output schemas
- Adapting tone and formatting conventions
- Simple domain shifts with clear demonstrations

It is insufficient for:
- Learning new factual knowledge absent from pretraining
- Tasks requiring specialized reasoning (e.g., medical image classification)
- Updating fundamental model capabilities

The model's fundamental capabilities are fixed by $\theta$. ICL leverages what $\theta$ already knows, directed by the context.

In [ ]:
categories_icl = [
    'Label format\nadaptation',
    'Output schema\n(JSON, CSV)',
    'Tone/style\nmatch',
    'Novel domain\nfacts',
    'Medical image\nclassification',
    'New reasoning\ncapabilities'
]

icl_benefit = [0.88, 0.82, 0.80, 0.30, 0.12, 0.18]
finetune_benefit = [0.90, 0.87, 0.85, 0.85, 0.82, 0.78]

x_pos = np.arange(len(categories_icl))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x_pos - width/2, icl_benefit, width, label='ICL (no param update)', color='steelblue', alpha=0.85)
ax.bar(x_pos + width/2, finetune_benefit, width, label='Fine-tuning', color='salmon', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(categories_icl, fontsize=9)
ax.set_ylabel('Relative benefit (schematic)')
ax.set_ylim(0, 1.0)
ax.set_title('ICL vs. fine-tuning: where each method helps')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axvline(2.5, color='gray', linestyle='--', alpha=0.5)
ax.text(1.0, 0.97, 'ICL competitive', ha='center', fontsize=9, color='steelblue')
ax.text(4.2, 0.97, 'Fine-tuning needed', ha='center', fontsize=9, color='salmon')
plt.tight_layout()
plt.show()

---
## 5. Supervised Fine-Tuning (SFT) and Instruction Tuning

### 5.1 The SFT Setup

**Instruction tuning** is a form of supervised fine-tuning in which the dataset consists of (instruction, response) pairs. Starting from a pretrained checkpoint $\theta$, the model is trained to predict $Y = (y_1, \ldots, y_T)$ given instruction $X = (x_1, \ldots, x_I)$:

$$\mathcal{L}_{\text{SFT}}(\theta) = -\frac{1}{N} \sum_{i=1}^{N} \sum_{t=1}^{T_i} \log p_\theta(y_t^{(i)} \mid x^{(i)}, y_{<t}^{(i)})$$

The instruction tokens $X$ are provided as context but **do not contribute to the loss** — loss is computed only over the response tokens $Y$.

### 5.2 Why Only Predict Y?

The instruction $X$ is treated as given (prefill): the model's job is to generate the response, not to reconstruct the prompt. Including the instruction in the loss can hurt performance by spreading gradient signal over non-target tokens.

### 5.3 Instruction Tuning vs. Standard SFT

| Aspect | Pretraining | Standard SFT | Instruction Tuning |
|--------|-------------|--------------|--------------------|
| Data | Raw text | Labeled (X, Y) pairs | (Instruction, Response) pairs |
| Loss target | All tokens | Y | Y only |
| Goal | Language model | Task-specific | General instruction following |

In [ ]:
def compute_sft_loss(logits_per_token, targets, instruction_mask):
    """
    logits_per_token : (T, V) — raw logits for each position
    targets          : (T,)   — target token indices
    instruction_mask : (T,)   — True for instruction positions (excluded from loss)
    Returns mean cross-entropy over response positions only.
    """
    T, V = logits_per_token.shape
    probs = np.exp(logits_per_token - logits_per_token.max(axis=-1, keepdims=True))
    probs /= probs.sum(axis=-1, keepdims=True)
    target_probs = probs[np.arange(T), targets]
    token_loss = -np.log(target_probs + 1e-12)
    response_mask = ~instruction_mask
    return token_loss[response_mask].mean(), token_loss


np.random.seed(0)
V = 32000
T_instruction = 12
T_response = 20
T_total = T_instruction + T_response

logits = np.random.randn(T_total, V) * 2.0
targets = np.random.randint(0, V, size=T_total)
instruction_mask = np.array([True] * T_instruction + [False] * T_response)

sft_loss, per_token_loss = compute_sft_loss(logits, targets, instruction_mask)

full_loss_all = per_token_loss.mean()
response_loss = per_token_loss[~instruction_mask].mean()
instruction_loss = per_token_loss[instruction_mask].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = ['#aec6cf'] * T_instruction + ['#f4a261'] * T_response
axes[0].bar(range(T_total), per_token_loss, color=colors)
axes[0].axvline(T_instruction - 0.5, color='red', linestyle='--', linewidth=1.5, label='Instruction | Response boundary')
axes[0].set_xlabel('Token position')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Per-token loss\n(blue=instruction, orange=response)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

bar_labels = ['Train on\nall tokens', 'Train on\nresponse only', 'Instruction\ntokens only']
bar_vals = [full_loss_all, response_loss, instruction_loss]
bar_colors = ['gray', '#f4a261', '#aec6cf']
axes[1].bar(bar_labels, bar_vals, color=bar_colors, alpha=0.85, edgecolor='black')
axes[1].set_ylabel('Mean cross-entropy')
axes[1].set_title('Mean loss under different masking strategies')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(bar_vals):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"SFT loss (response tokens only): {sft_loss:.4f}")
print(f"Loss if computed over all tokens: {full_loss_all:.4f}")
print(f"Instruction token loss (excluded from training): {instruction_loss:.4f}")

### 5.4 Training Dynamics: SFT Loss Curve

A typical instruction-tuning run starts from a pretrained checkpoint. The loss drops quickly in early steps as the model learns the instruction-following format, then levels off. Overfitting on small instruction datasets is a common problem, motivating regularization techniques like low-rank adaptation (LoRA) or early stopping.

In [ ]:
np.random.seed(1)

steps = np.arange(1, 501)

def loss_curve(steps, L0, L_inf, decay_rate, noise_std):
    smooth = L_inf + (L0 - L_inf) * np.exp(-decay_rate * steps)
    noise = np.random.randn(len(steps)) * noise_std
    return smooth + noise

train_loss = loss_curve(steps, 2.8, 0.45, 0.015, 0.03)
val_loss_small = loss_curve(steps, 2.9, 0.52, 0.012, 0.04)
val_loss_small[250:] += np.linspace(0, 0.35, 250)
val_loss_large = loss_curve(steps, 2.9, 0.48, 0.013, 0.04)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(steps, train_loss, label='Train loss', color='steelblue')
axes[0].plot(steps, val_loss_small, label='Val loss (small dataset)', color='salmon')
axes[0].plot(steps, val_loss_large, label='Val loss (large dataset)', color='green', linestyle='--')
axes[0].set_xlabel('Training steps')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('SFT training dynamics')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

dataset_sizes = [100, 500, 1000, 5000, 20000, 100000]
final_val = [1.85, 1.40, 1.10, 0.75, 0.58, 0.50]
axes[1].semilogx(dataset_sizes, final_val, 'o-', color='steelblue', linewidth=2)
axes[1].set_xlabel('Instruction dataset size (log scale)')
axes[1].set_ylabel('Final val loss')
axes[1].set_title('Instruction dataset size vs. final performance')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

| Technique | Problem solved | Mechanism | Tradeoff |
|-----------|---------------|-----------|----------|
| **GQA** | KV cache memory bottleneck | Share $N_G \ll N_H$ key/value groups across query heads | Reduced model capacity vs. MHA |
| **Sliding Window** | Quadratic attention compute | Each query attends to only $W$ recent positions | Maximum context depth = $L \times W$ |
| **MoE** | Scaling parameters without scaling compute | Route each token to $K$ of $M$ expert MLPs | Load balancing required; all params in memory |
| **ICL** | Deployment without fine-tuning | Encode task in context string; no parameter update | Limited to what pretrained model already knows |
| **SFT / Instruction Tuning** | Model alignment to instruction-following | Fine-tune on (instruction, response) pairs with loss only on response | Risk of overfitting on small datasets |

These five techniques are orthogonal and are routinely combined: a GQA + MoE transformer trained with instruction tuning and deployed with zero-shot or few-shot prompting is the architecture pattern of most frontier models.